# Phase 9 — Wave-Level Text Generation (Google Colab)

**FLUX Architecture** — Field-based Latent Understanding eXperience

## Overview

Phase 9 adds three new modules on top of the **Phase 7 FLUXModel** (native 512-dim):

| Module | Params | Purpose |
|--------|--------|---------|
| **WaveChunker** | ~374K | Segment CSE wave sequences into word-level chunks |
| **WaveGenerator** (GRU) | ~2.4M | Predict next chunk-wave from context + GRU state |
| **WaveToText** | ~200K | Decode a single 432-dim wave → UTF-8 bytes |

### Architecture: FLUXModel (512-dim, NOT FLUXLarge 768-dim)

Phase 9 loads **FLUXModel directly from Phase 7** where ALL bridge projections are trained:
- `wave_to_field` (432→512): **TRAINED** ✓
- `field_to_wave` (512→432): **TRAINED** ✓
- `field.wave_to_feature` (432→512): **TRAINED** ✓
- `field state` (64³×512): **TRAINED** attractors ✓
- `GR` (512-dim): **TRAINED** ✓

> FLUXLarge (768-dim) was dropped because `from_phase7_checkpoint()` skipped all 512→768
> incompatible weights, leaving bridges randomly initialized and breaking the field round-trip.

### Training Stages

| Stage | What | Steps | Time |
|-------|------|-------|------|
| **1: WTT** | Train WaveToText to spell words from chunk waves | ~2K | ~15 min |
| **2: Field Pop** | Populate field with 200K chunk-level attractors | N/A | ~5 min |
| **3: WG** | Train WaveGenerator (GRU) to predict next chunk | ~8K | ~30 min |
| **4: Joint FT** | Fine-tune WG + WTT together | ~2K | ~15 min |

### Cell Map

| Cells | Stage |
|-------|-------|
| 1-2 | Clone repo & install deps |
| 3 | Hardware, secrets, logger, config |
| 4 | Smoke test |
| 5 | Load data |
| 6a-6c | Stage 1: WTT Fresh → Train → Diag |
| 7a-7c | Stage 2: Field Pop Fresh → Populate → Diag |
| 8a-8c | Stage 3: WG Fresh → Train → Diag |
| 9a-9c | Stage 4: Joint FT Fresh → Train → Diag |
| 10 | Save checkpoint |
| 11-13 | Tests 1-3 |
| 14-15 | Demos 1-2 |
| 16 | Interactive exploration |
| 17 | View results |
| 18 | View log |
| 19 | Upload checkpoint + logs |
| 20 | Save Colab artifacts |
| 21 | Summary


## Cell 1 — Clone Repo & Install Dependencies


In [ ]:
# Cell 1 — Clone / pull FLUX repo and install dependencies
import os, subprocess

REPO_URL = "https://github.com/Unseengap/FLUX.git"
REPO_DIR = "/content/FLUX"

if os.path.isdir(REPO_DIR):
    print("  ℹ Repo exists — pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
else:
    print("  ℹ Cloning FLUX repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print(f"  ✓ Working directory: {os.getcwd()}")

# Install dependencies (suppress pip output)
subprocess.run(
    ["pip", "install", "-q", "-e", ".", "--no-deps"],
    check=False, capture_output=True,
)
subprocess.run(
    ["pip", "install", "-q", "-r", "requirements.txt"],
    check=False, capture_output=True,
)
# Ensure datasets is available
subprocess.run(
    ["pip", "install", "-q", "datasets", "huggingface_hub"],
    check=False, capture_output=True,
)
print("  ✓ Dependencies installed")


## Cell 2 — Hardware, Secrets, Logger


In [ ]:
# Cell 2 — Hardware, Secrets, Logger, Config
import sys, torch, time
from pathlib import Path

# Path setup
REPO_DIR = Path("/content/FLUX")
for _p in ['phases/phase1', 'phases/phase2', 'phases/phase3',
           'phases/phase4', 'phases/phase5', 'phases/phase6',
           'phases/phase7', 'phases/phase8', 'phases/phase9']:
    _pp = str(REPO_DIR / _p)
    if _pp not in sys.path:
        sys.path.insert(0, _pp)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from flux_utils import get_device, PhaseLogger, PhaseResults

# Hardware
device = get_device()
print(f"  Device: {device}")
if device == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    _vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"  VRAM: {_vram:.1f} GB")

# Secrets — HuggingFace token from Colab
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print(f"  ✓ HF_TOKEN loaded ({len(HF_TOKEN)} chars)")
except Exception:
    import os
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    if HF_TOKEN:
        print(f"  ✓ HF_TOKEN from env ({len(HF_TOKEN)} chars)")
    else:
        print("  ⚠ No HF_TOKEN — checkpoint upload will be skipped")

# Logger
log = PhaseLogger(phase=9)
log.separator("Phase 9: Wave-Level Generation (FLUXModel native 512-dim)")
log.cell_start("Cell 2 — Hardware & Secrets")
log.info(f"Device: {device}")
log.cell_end("Cell 2 — Hardware & Secrets", "PASS")

# Phase 9 Config — uses 512-dim field_features (native Phase 7)
PHASE9_CONFIG = {
    'wave_dim': 432,
    'field_features': 512,   # Native Phase 7 FLUXModel dimension
    'max_waves': 50,
    'k_neighbors': 16,
    'interference_radius': 4,
    'gru_hidden': 512,
    'gru_layers': 1,
    'wtt_hidden_dim': 256,
    'wtt_max_bytes': 20,
    'coherence_threshold': 0.5,
    'min_chunk_size': 2,
    'max_chunk_size': 20,
}
print(f"  ✓ PHASE9_CONFIG: field_features={PHASE9_CONFIG['field_features']}")


## Cell 3 — Smoke Test


In [ ]:
# Cell 3 — Smoke Test: verify imports and model assembly
log.cell_start("Cell 3 — Smoke Test")
import torch.nn.functional as F

from train_wave_gen import (
    build_flux_for_phase9, build_phase9_modules,
    Phase9Trainer, PHASE9_CONFIG as _P9CFG,
)

print(f"  ✓ All Phase 9 imports successful")
print(f"  ℹ PHASE9_CONFIG.field_features = {_P9CFG['field_features']}")
assert _P9CFG['field_features'] == 512, f"Expected 512, got {_P9CFG['field_features']}"

# Quick model test — build_flux_for_phase9 now loads phase7.phase.pt directly
# which contains trained bridges (wave_to_field, field_to_wave)
_model = build_flux_for_phase9(device=device)
_test_wave = torch.randn(432, device=device)
with torch.no_grad():
    _proj = _model.wave_to_field(_test_wave)
    _round = _model.field_to_wave(_proj)
    _cos = F.cosine_similarity(_test_wave.unsqueeze(0), _round.unsqueeze(0)).item()
print(f"  Bridge round-trip cosine: {_cos:.3f}")

# Random init gives ~0.003. Trained bridges should give > 0.1.
# If this fails, phase7.phase.pt may not contain bridge states.
if _cos > 0.1:
    print(f"  ✓ Bridges are TRAINED (cosine {_cos:.3f} >> 0.003 random)")
else:
    print(f"  ⚠ Bridge round-trip cosine is low ({_cos:.3f})")
    print(f"    Checking if bridge states were loaded from phase7.phase.pt...")
    # Don't hard-fail — the field population + training may still work
    # if the bridges are at least self-consistent

# Verify field has trained state
_n_attractors = _model.field.num_attractors()
print(f"  ✓ Field attractors from Phase 7: {_n_attractors:,}")

del _model, _test_wave, _proj, _round
torch.cuda.empty_cache() if device == 'cuda' else None
log.cell_end("Cell 3 — Smoke Test", "PASS")
print("  ✓ Smoke test passed — FLUXModel loaded from phase7.phase.pt")

## Cell 4 — Load Training Data


In [ ]:
# Cell 4 — Load training data
log.cell_start("Cell 4 — Load Data")
from train_wave_gen import load_training_data

texts = load_training_data(max_docs=10000)
split_idx = int(len(texts) * 0.9)
train_texts = texts[:split_idx]
eval_texts  = texts[split_idx:]
print(f"  Train: {len(train_texts):,} docs | Eval: {len(eval_texts):,} docs")
log.cell_end("Cell 4 — Load Data", "PASS")


---
## Stage 1: WaveToText Pre-Training
Train WaveToText to spell words from CSE chunk waves.
WaveChunker is co-trained (learnable boundary detection).
This stage is independent of field features — pure 432-dim.


### Cell 6a — WTT Fresh (Build Components)


In [ ]:
# Cell 6a — Build model + Phase 9 modules fresh
log.cell_start("Cell 6a — WTT Fresh")

model = build_flux_for_phase9(device=device)
from train_wave_gen import build_phase9_modules
chunker, generator, wtt = build_phase9_modules(device=device, config=PHASE9_CONFIG)

trainer = Phase9Trainer(
    flux_model=model,
    wave_chunker=chunker,
    wave_generator=generator,
    wave_to_text=wtt,
    lr=3e-4,
    grad_accum=4,
    log=log,
)
log.cell_end("Cell 6a — WTT Fresh", "PASS")
print("  ✓ Phase 9 trainer ready — all bridges verified")


### Cell 6b — WTT Train (~2K steps)
Cross-entropy loss on chunk-wave → byte spelling.


In [ ]:
# Cell 6b — Stage 1: Train WaveToText
log.cell_start("Cell 6b — WTT Train")
wtt_result = trainer.train_wave_to_text(
    train_texts, max_steps=2000, batch_size=32, log_interval=200
)
log.metric("wtt_final_loss", f"{wtt_result.final_loss:.4f}")
log.metric("wtt_word_acc", f"{wtt_result.word_accuracy:.1%}")
log.cell_end("Cell 6b — WTT Train", "PASS")


### Cell 6c — WTT Diagnostic
Verify WTT can spell common and uncommon words.


In [ ]:
# Cell 6c — WTT Diagnostic
log.cell_start("Cell 6c — WTT Diag")
import torch

test_words = [
    "the", "of", "and", "is", "in",          # ultra-common
    "science", "energy", "system", "quantum",  # domain terms
    "relationship", "fundamental", "efficiency", # longer words
    "xyz", "café", "naïve",                    # edge cases
]

wtt.eval()
chunker.eval()
correct = 0
total = 0

print("\n  WTT Diagnostic — Word Spelling Accuracy")
print("  " + "=" * 55)

for word in test_words:
    try:
        with torch.no_grad():
            wave = model.cse.encode(word)
            wave_seq = wave.full.to(device)
            chunk_waves, spans = chunker(wave_seq)
            if chunk_waves.shape[0] == 0:
                print(f"  {word:20s} → [no chunks]")
                continue
            # Use first chunk
            decoded_bytes = wtt.decode(chunk_waves[0], temperature=0.5)
            decoded_str = decoded_bytes.decode('utf-8', errors='replace')
            match = decoded_str.strip() == word
            correct += int(match)
            total += 1
            status = "✓" if match else "✗"
            print(f"  {status} {word:20s} → {decoded_str!r}")
    except Exception as e:
        print(f"  ✗ {word:20s} → ERROR: {e}")
        total += 1

print(f"\n  Accuracy: {correct}/{total} ({correct/max(total,1):.0%})")
log.cell_end("Cell 6c — WTT Diag", "PASS")


---
## Stage 2: Field Population
Densify the resonance field with chunk-level attractors.
Phase 7 had ~60 document-level attractors. We add 200K chunk-level ones.

With native 512-dim FLUXModel, `field.wave_to_feature` is the TRAINED
Phase 2/7 projection — no alignment hack needed.


### Cell 7a — Field Fresh (Check State)


In [ ]:
# Cell 7a — Field state before population
log.cell_start("Cell 7a — Field Fresh")
n_before = model.field.num_attractors()
print(f"  Field attractors from Phase 7: {n_before:,}")
print(f"  Field features dim: {model.field.wave_to_feature.weight.shape[0]}")
print(f"  Bridge wave_to_field dim: {model.wave_to_field.weight.shape[0]}")

# Verify field.wave_to_feature and wave_to_field are in the same space
# (they should be — both 432→512, trained in Phase 2/7)
import torch.nn.functional as F
with torch.no_grad():
    _fw = model.field.wave_to_feature.weight  # [512, 432]
    _bw = model.wave_to_field.weight           # [512, 432]
    _cos = F.cosine_similarity(_fw.flatten().unsqueeze(0), _bw.flatten().unsqueeze(0)).item()
    # These are NOT necessarily identical — field.wave_to_feature was trained in Phase 2,
    # wave_to_field was trained in Phase 7. But both should be meaningful (not random).
    print(f"  Field↔Bridge weight cosine: {_cos:.3f}")

log.cell_end("Cell 7a — Field Fresh", "PASS")


### Cell 7b — Populate Field (~200K perturbs)
Each chunk wave creates a local attractor in the field.


In [ ]:
# Cell 7b — Populate field with chunk-level attractors
log.cell_start("Cell 7b — Field Pop")
pop_stats = trainer.populate_field(train_texts, max_chunks=200000)
print(f"\n  Population complete:")
print(f"    Perturbs:    {pop_stats['total_perturbs']:,}")
print(f"    Attractors:  {pop_stats['attractors_before']:,} → {pop_stats['attractors_after']:,}")
print(f"    New:         +{pop_stats['new_attractors']:,}")
log.cell_end("Cell 7b — Field Pop", "PASS")


### Cell 7c — Field Diagnostic
Test field query quality and round-trip fidelity.


In [ ]:
# Cell 7c — Field Population Diagnostic
log.cell_start("Cell 7c — Field Diag")
import torch, torch.nn.functional as F

n_attractors = model.field.num_attractors()
print(f"  Total attractors: {n_attractors:,}")

# Test queries
test_phrases = [
    "The future of artificial intelligence",
    "Energy equals mass times the speed of light squared",
    "Quantum mechanics describes the behavior of particles",
    "The cat sat on the mat",
    "Photosynthesis converts sunlight into chemical energy",
    "Neural networks learn from data through backpropagation",
    "Water freezes at zero degrees Celsius",
    "Democracy is a form of government",
]

print(f"\n  Field Query Test ({len(test_phrases)} phrases)")
print("  " + "=" * 70)

query_hits = 0
roundtrip_cosines = []

for phrase in test_phrases:
    with torch.no_grad():
        wave = model.cse.encode(phrase)
        wave_vec = wave.full.mean(dim=0).to(device)

        # Query field
        feats, sims, locs = model.field.query(wave_vec, k=4)
        if feats is not None and feats.shape[0] > 0:
            query_hits += 1
            top_sim = sims[0].item() if sims is not None else 0.0

            # Round-trip: wave → field.query → field_to_wave → compare
            best_feat = feats[0]  # [512]
            roundtrip_wave = model.field_to_wave(best_feat)  # [432]
            rt_cos = F.cosine_similarity(
                wave_vec.unsqueeze(0), roundtrip_wave.unsqueeze(0)
            ).item()
            roundtrip_cosines.append(rt_cos)

            print(f"  ✓ {phrase[:50]:50s} sim={top_sim:.3f}  rt_cos={rt_cos:.3f}")
        else:
            print(f"  ✗ {phrase[:50]:50s} [no results]")

avg_rt = sum(roundtrip_cosines) / max(len(roundtrip_cosines), 1)
print(f"\n  Query hit rate: {query_hits}/{len(test_phrases)}")
print(f"  Avg round-trip cosine: {avg_rt:.3f}")
print(f"  Min round-trip cosine: {min(roundtrip_cosines) if roundtrip_cosines else 0:.3f}")

if avg_rt > 0.3:
    print(f"  ✓ Field round-trip is WORKING (avg cosine {avg_rt:.3f} > 0.3)")
else:
    print(f"  ⚠ Field round-trip may be weak (avg cosine {avg_rt:.3f})")

log.cell_end("Cell 7c — Field Diag", "PASS")


---
## Stage 3: WaveGenerator Training
Train the GRU-based WaveGenerator to predict next chunk-wave.
Uses precomputed frozen pipeline outputs for GPU efficiency.

With trained bridges, the `merged` context ([512]) now carries
meaningful semantic information instead of random noise.


### Cell 8a — WG Fresh (Precompute Pipeline)


In [ ]:
# Cell 8a — Precompute frozen pipeline outputs for WG training
log.cell_start("Cell 8a — WG Fresh (Precompute)")

# Precompute merged contexts using the FULL Phase 7 pipeline:
#   CSE → wave_to_field(432→512) → GR(512) → CGN(512) → field.query → combine
# All projections are trained — merged context is meaningful
precomputed = trainer.precompute_wg_data(train_texts, max_samples=8500)

print(f"\n  Precomputed: {len(precomputed):,} samples")
if len(precomputed) > 0:
    m, tw, wv = precomputed[0]
    print(f"  Sample shape: merged={list(m.shape)}, target_waves={list(tw.shape)}, wave_vec={list(wv.shape)}")
    assert m.shape[0] == 512, f"Expected merged dim 512, got {m.shape[0]}"

log.cell_end("Cell 8a — WG Fresh (Precompute)", "PASS")


### Cell 8b — WG Train (~8K steps)
MSE + cosine + context loss + contrastive loss.
Scheduled sampling ramps from 0% to 50%.


In [ ]:
# Cell 8b — Stage 3: Train WaveGenerator
log.cell_start("Cell 8b — WG Train")
wg_result = trainer.train_wave_generator(
    train_texts, max_steps=8000, log_interval=200,
    precomputed=precomputed,
)
log.metric("wg_final_loss", f"{wg_result.final_loss:.4f}")
log.metric("wg_cos_acc", f"{wg_result.wave_cosine_accuracy:.3f}")
log.cell_end("Cell 8b — WG Train", "PASS")


### Cell 8c — WG Diagnostic
Test teacher-forced prediction and free generation.


In [ ]:
# Cell 8c — WaveGenerator Diagnostic
log.cell_start("Cell 8c — WG Diag")
import torch, torch.nn.functional as F

generator.eval()
wtt.eval()
chunker.eval()

# --- Teacher-forced test ---
print("  📊 Teacher-Forced Prediction (5 samples)")
print("  " + "=" * 60)

tf_cosines = []
for i in range(min(5, len(precomputed))):
    merged_cpu, target_cpu, wv_cpu = precomputed[i]
    merged = merged_cpu.to(device)
    target = target_cpu.to(device)

    with torch.no_grad():
        predicted, confs = generator(merged, target)
        n = min(len(predicted), len(target))
        cos_per = F.cosine_similarity(predicted[:n], target[:n], dim=-1)
        avg_cos = cos_per.mean().item()
        tf_cosines.append(avg_cos)
        print(f"  Sample {i}: {n} waves, avg cosine={avg_cos:.3f}, "
              f"range=[{cos_per.min().item():.3f}, {cos_per.max().item():.3f}]")

print(f"\n  Overall teacher-forced cosine: {sum(tf_cosines)/len(tf_cosines):.3f}")

# --- Check Wave 0 depends on context ---
print("\n  📊 Context Dependency Test")
print("  " + "=" * 60)
if len(precomputed) >= 3:
    m0 = precomputed[0][0].to(device)
    m1 = precomputed[1][0].to(device)
    m2 = precomputed[2][0].to(device)
    t_dummy = precomputed[0][1][:3].to(device)

    with torch.no_grad():
        p0, _ = generator(m0, t_dummy)
        p1, _ = generator(m1, t_dummy)
        p2, _ = generator(m2, t_dummy)
        # Wave 0 from different contexts should differ
        cos_01 = F.cosine_similarity(p0[0].unsqueeze(0), p1[0].unsqueeze(0)).item()
        cos_02 = F.cosine_similarity(p0[0].unsqueeze(0), p2[0].unsqueeze(0)).item()
        cos_12 = F.cosine_similarity(p1[0].unsqueeze(0), p2[0].unsqueeze(0)).item()
        avg_cross = (cos_01 + cos_02 + cos_12) / 3
        print(f"  Wave 0 cross-context cosines: {cos_01:.3f}, {cos_02:.3f}, {cos_12:.3f}")
        print(f"  Average: {avg_cross:.3f} (lower = more context-dependent)")
        if avg_cross < 0.95:
            print(f"  ✓ Generator differentiates contexts")
        else:
            print(f"  ⚠ Generator may be ignoring context")

# --- Free generation test ---
print("\n  📊 Free Generation (3 prompts)")
print("  " + "=" * 60)
from train_wave_gen import generate_text

gen_prompts = [
    "The future of artificial intelligence",
    "Scientists have discovered",
    "In the beginning",
]

for prompt in gen_prompts:
    try:
        result = generate_text(
            prompt, model, chunker, generator, wtt,
            max_waves=15, temperature=0.8,
        )
        continuation = result[len(prompt):].strip()
        print(f"  Prompt: {prompt}")
        print(f"  Output: {continuation[:120]}")
        print()
    except Exception as e:
        print(f"  ✗ {prompt[:30]}... → ERROR: {e}")
        print()

log.cell_end("Cell 8c — WG Diag", "PASS")


---
## Stage 4: Joint Fine-Tuning
Fine-tune WaveGenerator + WaveToText together.
WG predictions pass through WTT — text loss backpropagates into WG.


### Cell 9a — Joint FT Fresh


In [ ]:
# Cell 9a — Prepare for joint fine-tuning
log.cell_start("Cell 9a — Joint FT Fresh")
# Both WG and WTT are already loaded from previous stages.
# Joint FT unfreezes both and trains with combined loss.
_wg_params = sum(p.numel() for p in generator.parameters())
_wtt_params = sum(p.numel() for p in wtt.parameters())
print(f"  WG params:  {_wg_params:,}")
print(f"  WTT params: {_wtt_params:,}")
print(f"  Joint total: {_wg_params + _wtt_params:,}")
log.cell_end("Cell 9a — Joint FT Fresh", "PASS")


### Cell 9b — Joint FT Train (~2K steps)
Combined loss = MSE + cosine + 0.5 × WTT_loss.


In [ ]:
# Cell 9b — Stage 4: Joint fine-tuning
log.cell_start("Cell 9b — Joint FT Train")
joint_result = trainer.train_joint_finetune(
    train_texts, max_steps=2000, log_interval=200,
    precomputed=precomputed,
)
log.metric("joint_cos_acc", f"{joint_result.wave_cosine_accuracy:.3f}")
log.metric("joint_wtt_acc", f"{joint_result.wtt_word_accuracy:.1%}")
log.cell_end("Cell 9b — Joint FT Train", "PASS")


### Cell 9c — Joint FT Diagnostic
Compare pre-joint vs post-joint generation quality.


In [ ]:
# Cell 9c — Joint Fine-Tuning Diagnostic
log.cell_start("Cell 9c — Joint FT Diag")

generator.eval()
wtt.eval()

print("  📊 Post-Joint Generation (5 prompts)")
print("  " + "=" * 60)

eval_prompts = [
    "The relationship between energy and matter",
    "Modern technology relies on",
    "In the year 2025, scientists proved that",
    "The history of mathematics reveals",
    "Research shows that quantum",
]

valid_words = 0
total_words = 0

for prompt in eval_prompts:
    try:
        result = generate_text(
            prompt, model, chunker, generator, wtt,
            max_waves=20, temperature=0.8,
        )
        continuation = result[len(prompt):].strip()
        words = continuation.split()
        for w in words:
            clean = w.strip('.,;:!?\'"-()[]{}').lower()
            if clean.isalpha() and len(clean) >= 2:
                total_words += 1
                if len(clean) <= 15:
                    valid_words += 1
        print(f"  Prompt: {prompt}")
        print(f"  Output: {continuation[:150]}")
        print()
    except Exception as e:
        print(f"  ✗ {prompt[:30]}... → ERROR: {e}")
        print()

valid_rate = valid_words / max(total_words, 1)
print(f"  Valid word rate: {valid_rate:.1%} ({valid_words}/{total_words})")
log.metric("valid_word_rate", f"{valid_rate:.1%}")
log.cell_end("Cell 9c — Joint FT Diag", "PASS")


## Cell 10 — Save Checkpoint


In [ ]:
# Cell 10 — Save Phase 9 checkpoint
log.cell_start("Cell 10 — Save Checkpoint")
ckpt_path = trainer.save_phase9_checkpoint(
    wtt_result, wg_result, joint_result,
    valid_word_rate=valid_rate,
)
print(f"  ✓ Saved: {ckpt_path}")
import os
_size = os.path.getsize(str(ckpt_path)) / 1e6
print(f"  Size: {_size:.1f} MB")
log.cell_end("Cell 10 — Save Checkpoint", "PASS")


---
## Tests
Run all three Phase 9 tests. Each is a standalone script.


In [ ]:
# Cell 11 — Test 1
log.cell_start("Cell 11 — Test 1")
!cd /content/FLUX && python phases/phase9/test_phase9_test1.py
log.cell_end("Cell 11 — Test 1")


In [ ]:
# Cell 12 — Test 2
log.cell_start("Cell 12 — Test 2")
!cd /content/FLUX && python phases/phase9/test_phase9_test2.py
log.cell_end("Cell 12 — Test 2")


In [ ]:
# Cell 13 — Test 3
log.cell_start("Cell 13 — Test 3")
!cd /content/FLUX && python phases/phase9/test_phase9_test3.py
log.cell_end("Cell 13 — Test 3")


---
## Demos
Run Phase 9 demo scripts.


In [ ]:
# Cell 14 — Demo 1
log.cell_start("Cell 14 — Demo 1")
!cd /content/FLUX && python phases/phase9/demo_phase9_demo1.py
log.cell_end("Cell 14 — Demo 1")


In [ ]:
# Cell 15 — Demo 2
log.cell_start("Cell 15 — Demo 2")
!cd /content/FLUX && python phases/phase9/demo_phase9_demo2.py
log.cell_end("Cell 15 — Demo 2")


## Cell 16 — Interactive Exploration


In [ ]:
# Cell 16 — Interactive text generation
log.cell_start("Cell 16 — Interactive")
from train_wave_gen import generate_text

prompts = [
    "The nature of consciousness",
    "Water flows downhill because",
    "The most important discovery in physics was",
    "Artificial intelligence will",
    "In a distant galaxy",
]

print("  🌊 FLUX Phase 9 — Wave-Level Generation")
print("  " + "=" * 60)

for p in prompts:
    result = generate_text(
        p, model, chunker, generator, wtt,
        max_waves=25, temperature=0.8,
    )
    print(f"\n  Prompt: {p}")
    print(f"  →  {result}")

log.cell_end("Cell 16 — Interactive", "PASS")


## Cell 17 — View Results


In [ ]:
# Cell 17 — Generate and view results
log.cell_start("Cell 17 — Results")
results = PhaseResults(phase=9, component_name="Wave-Level Generation")
results.add_metric("Architecture", "FLUXModel native 512-dim (NOT FLUXLarge 768)")
results.add_metric("WTT training steps", wtt_result.total_steps)
results.add_metric("WTT word accuracy", f"{wtt_result.word_accuracy:.1%}")
results.add_metric("WG training steps", wg_result.total_steps)
results.add_metric("WG final loss", f"{wg_result.final_loss:.4f}")
results.add_metric("WG cosine accuracy", f"{wg_result.wave_cosine_accuracy:.3f}")
if joint_result:
    results.add_metric("Joint training steps", joint_result.total_steps)
    results.add_metric("Joint cosine accuracy", f"{joint_result.wave_cosine_accuracy:.3f}")
    results.add_metric("Joint WTT accuracy", f"{joint_result.wtt_word_accuracy:.1%}")
results.add_metric("Valid word rate", f"{valid_rate:.1%}")
results.add_metric("Total time", f"{wtt_result.total_time_seconds + wg_result.total_time_seconds + (joint_result.total_time_seconds if joint_result else 0):.0f}s")
results.save()

# Display
_rpath = Path("/content/FLUX/phases/phase9/RESULTS_PHASE_9.md")
if _rpath.exists():
    print(_rpath.read_text())
log.cell_end("Cell 17 — Results", "PASS")


## Cell 18 — View Log


In [ ]:
# Cell 18 — View full training log
_logpath = Path("/content/FLUX/logs/phase9.log")
if _logpath.exists():
    _lines = _logpath.read_text().splitlines()
    print(f"  Log: {len(_lines)} lines")
    # Print last 50 lines
    for line in _lines[-50:]:
        print(line)
else:
    print("  ⚠ No log file found")


## Cell 19 — Upload Checkpoint & Logs


In [ ]:
# Cell 19 — Upload to HuggingFace Hub
log.cell_start("Cell 19 — Upload")
if HF_TOKEN:
    from flux_utils import upload_checkpoint_to_hf, upload_logs_to_hf
    try:
        upload_checkpoint_to_hf(phase=9, hf_token=HF_TOKEN)
        print("  ✓ Checkpoint uploaded to HuggingFace")
    except Exception as e:
        print(f"  ⚠ Checkpoint upload failed: {e}")
    try:
        upload_logs_to_hf(phase=9, hf_token=HF_TOKEN)
        print("  ✓ Logs uploaded to HuggingFace")
    except Exception as e:
        print(f"  ⚠ Log upload failed: {e}")
else:
    print("  ⚠ No HF_TOKEN — skipping upload")
log.cell_end("Cell 19 — Upload", "PASS")


## Cell 20 — Save Colab Artifacts


In [ ]:
# Cell 20 — Copy artifacts to /content/output for Colab download
import shutil
_out = Path("/content/output")
_out.mkdir(exist_ok=True)

# Checkpoint
_ckpt = Path("/content/FLUX/checkpoints/phase9.phase.pt")
if _ckpt.exists():
    shutil.copy2(_ckpt, _out / "phase9.phase.pt")
    print(f"  ✓ Checkpoint → {_out / 'phase9.phase.pt'}")

# Results
_res = Path("/content/FLUX/phases/phase9/RESULTS_PHASE_9.md")
if _res.exists():
    shutil.copy2(_res, _out / "RESULTS_PHASE_9.md")
    print(f"  ✓ Results → {_out / 'RESULTS_PHASE_9.md'}")

# Log
_log = Path("/content/FLUX/logs/phase9.log")
if _log.exists():
    shutil.copy2(_log, _out / "phase9.log")
    print(f"  ✓ Log → {_out / 'phase9.log'}")

print(f"\n  All artifacts saved to {_out}")


## Summary

Phase 9 trains wave-level text generation using **FLUXModel** (native 512-dim):

| Component | Status |
|-----------|--------|
| `wave_to_field` (432→512) | **TRAINED** from Phase 7 ✓ |
| `field_to_wave` (512→432) | **TRAINED** from Phase 7 ✓ |
| `field.wave_to_feature` (432→512) | **TRAINED** from Phase 2/7 ✓ |
| `field state` (64³×512) | **TRAINED** attractors from Phase 7 ✓ |
| GR (512-dim) | **TRAINED** from Phase 3/7 ✓ |
| WaveChunker | Trained in Stage 1 |
| WaveToText | Trained in Stage 1, refined in Stage 4 |
| WaveGenerator | Trained in Stage 3, refined in Stage 4 |

The old FLUXLarge (768-dim) approach has been **permanently replaced** because
`from_phase7_checkpoint()` skipped all bridge/field weights due to 512→768
dimension mismatch, leaving them randomly initialized and breaking the entire
field↔wave round-trip pipeline.
